In [1]:
!pip install bayesian-torch
!pip install curvlinops-for-pytorch==2.0
!pip install laplace-torch
!pip install torchmetrics
!pip install matplotlib
!pip install pandas

  Using cached bayesian_torch-0.5.0-py3-none-any.whl.metadata (12 kB)
  Using cached torch-2.13.0-cp314-cp314-win_amd64.whl.metadata (39 kB)
  Using cached torchvision-0.28.0-cp314-cp314-win_amd64.whl.metadata (5.6 kB)
  Using cached tensorboard-2.21.0-py3-none-any.whl.metadata (1.8 kB)
  Using cached scikit_learn-1.9.0-cp314-cp314-win_amd64.whl.metadata (11 kB)
  Using cached joblib-1.5.3-py3-none-any.whl.metadata (5.5 kB)
  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
  Using cached absl_py-2.5.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached grpcio-1.83.0-cp314-cp314-win_amd64.whl.metadata (3.8 kB)
  Using cached markdown-3.10.3-py3-none-any.whl.metadata (5.1 kB)
  Using cached pillow-12.3.0-cp314-cp314-win_amd64.whl.metadata (9.3 kB)
  Using cached setuptools-84.0.0-py3-none-any.whl.metadata (6.6 kB)
  Using cached tensorboard_data_server-0.7.2-py3-none-any.whl.metadata (1.1 kB)
  Using cached werkzeug-3.1.8-py3-none-any.whl.metadata (4.0 kB)
  Using cached


[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


  Using cached curvlinops_for_pytorch-2.0.0-py3-none-any.whl.metadata (4.8 kB)
  Using cached backpack_for_pytorch-1.7.1-py3-none-any.whl.metadata (4.4 kB)
  Using cached tqdm-4.70.0-py3-none-any.whl.metadata (57 kB)
  Using cached einops-0.8.2-py3-none-any.whl.metadata (13 kB)
  Using cached einconv-0.1.0-py3-none-any.whl.metadata (1.9 kB)
  Using cached unfoldNd-0.2.3-py3-none-any.whl.metadata (1.5 kB)
Using cached curvlinops_for_pytorch-2.0.0-py3-none-any.whl (67 kB)
Using cached backpack_for_pytorch-1.7.1-py3-none-any.whl (196 kB)
Using cached einops-0.8.2-py3-none-any.whl (65 kB)
Using cached tqdm-4.70.0-py3-none-any.whl (80 kB)
Using cached unfoldNd-0.2.3-py3-none-any.whl (16 kB)
Using cached einconv-0.1.0-py3-none-any.whl (27 kB)

   ------ --------------------------------- 1/6 [einops]
   ------------- -------------------------- 2/6 [unfoldNd]
   -------------------- ------------------- 3/6 [einconv]
   -------------------------- ------------- 4/6 [backpack-for-pytorch]
   ----


[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


  Using cached laplace_torch-0.2.2.2-py3-none-any.whl.metadata (5.1 kB)
  Using cached asdfghjkl-0.1a4-py3-none-any.whl.metadata (3.2 kB)
  Using cached opt_einsum-3.4.0-py3-none-any.whl.metadata (6.3 kB)
  Using cached torchmetrics-1.9.0-py3-none-any.whl.metadata (23 kB)
  Using cached lightning_utilities-0.15.3-py3-none-any.whl.metadata (5.5 kB)
Using cached laplace_torch-0.2.2.2-py3-none-any.whl (77 kB)
Using cached asdfghjkl-0.1a4-py3-none-any.whl (89 kB)
Using cached opt_einsum-3.4.0-py3-none-any.whl (71 kB)
Using cached torchmetrics-1.9.0-py3-none-any.whl (983 kB)
Using cached lightning_utilities-0.15.3-py3-none-any.whl (31 kB)

   ---------------------------------------- 0/5 [opt_einsum]
   ---------------- ----------------------- 2/5 [torchmetrics]
   ---------------- ----------------------- 2/5 [torchmetrics]
   ---------------- ----------------------- 2/5 [torchmetrics]
   ---------------- ----------------------- 2/5 [torchmetrics]
   ---------------- ----------------------- 


[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader,Dataset
from laplace import Laplace
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

print("Library Versions:")
print('numpy:',np.__version__)
print('pandas:',pd.__version__)
print('torch:',torch.__version__)


Library Versions:
numpy: 2.5.2
pandas: 3.0.5
torch: 2.13.0+cpu


In [5]:

n_epochs = 200
verbose_option = True

# Regression for Naval Plant Maintenance

Load dataset

In [6]:
class DummyDataset(Dataset):

    def __init__(self, X, y):
        self.X = X
        self.y = y

    def __len__(self):
        return self.X.shape[0]

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

In [7]:
npm = pd.read_csv('navalplantmaintenance.csv',header=None)
npm_train, npm_test = train_test_split(npm,test_size=0.25,random_state=42)
npm_train_np = npm_train.to_numpy()
npm_test_np = npm_test.to_numpy()
x_train_np = npm_train_np[:,:16]
x_test_np = npm_test_np[:,:16]
y_train_np = npm_train_np[:,17]
y_test_np = npm_test_np[:,17]
x_mu = x_train_np.mean(axis=0)
x_sigma = x_train_np.std(axis=0)
y_mu=y_train_np.mean(axis=0)
y_sigma=y_train_np.std(axis=0)
def scale(x,x_mu,x_sigma):
  x_sigma += 1e-16 #This is to deal with constant or near-constant columns
  return (x-x_mu)/x_sigma
def unscale(x,x_mu,x_sigma):
  x_sigma += 1e-16 #This is to deal with constant or near-constant columns
  return x_sigma*x+x_mu
x_train_np_z = scale(x_train_np,x_mu,x_sigma)
y_train_np_z = scale(y_train_np,y_mu,y_sigma)
x_test_np_z = scale(x_test_np,x_mu,x_sigma)
y_test_np_z = scale(y_test_np,y_mu,y_sigma)
x_train_t_z = torch.FloatTensor(x_train_np_z)
y_train_t_z = torch.FloatTensor(y_train_np_z)
x_test_t_z = torch.FloatTensor(x_test_np_z)
y_test_t_z = torch.FloatTensor(y_test_np_z)

In [8]:
train_t_z_dataset = DummyDataset(x_train_t_z,y_train_t_z)
train_t_z_dataloader = DataLoader(train_t_z_dataset)

1. Using PyTorch, perform variational inference using MC Dropout for a non-linear Gaussian
prediction model with heteroscedastic uncertainty for the regression dataset.

In [9]:
def nlls(y, mu, std):
    return torch.square(y - mu)/(2.0*torch.square(std))+torch.log(std)
n_train_examples = x_train_np_z.shape[0]

class nn(torch.nn.Module):
    def __init__(self, inputSize, hiddenSize, outputSize):
        super(nn, self).__init__()
        self.layer1 = torch.nn.Linear(inputSize, hiddenSize)
        self.layer2 = torch.nn.Linear(hiddenSize, hiddenSize)
        self.linear_mu = torch.nn.Linear(hiddenSize, outputSize)
        self.linear_sigma = torch.nn.Linear(hiddenSize, outputSize)

    def forward(self, x):
        h1 = torch.nn.functional.relu(self.layer1(torch.nn.functional.dropout(x, p=0.8, training=True))) # Run the first layer using dropout with p_drop = 0.8 and ReLU
        h2 = torch.nn.functional.relu(self.layer2(torch.nn.functional.dropout(h1, p=0.8, training=True))) # Run the second layer using dropout with p_drop = 0.8 and ReLU
        mu = self.linear_mu(h2)
        sigma = torch.nn.functional.softplus(self.linear_sigma(h2))
        return mu, sigma

    def L2reg(self):
        l2reg_sum = 0.0
        l2reg_sum += torch.square(self.layer1.weight).sum()
        l2reg_sum += torch.square(self.layer1.bias).sum()
        l2reg_sum += torch.square(self.layer2.weight).sum()
        l2reg_sum += torch.square(self.layer2.bias).sum()
        l2reg_sum += torch.square(self.linear_mu.weight).sum()
        l2reg_sum += torch.square(self.linear_mu.bias).sum()
        l2reg_sum += torch.square(self.linear_sigma.weight).sum()
        l2reg_sum += torch.square(self.linear_sigma.bias).sum()
        return l2reg_sum

In [10]:
model = nn(x_train_np_z.shape[1],50, 1)
optimizer = torch.optim.Adam(params=model.parameters(), lr=0.01)
l2_coeff = 1e-4
for i in range(50):
    mu,s= model(x_train_t_z)
    nll_loss = nlls(y_train_t_z, mu, s).mean()
    loss = nll_loss + l2_coeff * model.L2reg()
    loss.backward()
    optimizer.step()
    optimizer.zero_grad()
    if verbose_option: print(i, loss)

0 tensor(3.7914, grad_fn=<AddBackward0>)
1 tensor(0.6745, grad_fn=<AddBackward0>)
2 tensor(0.6579, grad_fn=<AddBackward0>)
3 tensor(0.6825, grad_fn=<AddBackward0>)
4 tensor(0.7107, grad_fn=<AddBackward0>)
5 tensor(0.7336, grad_fn=<AddBackward0>)
6 tensor(0.7423, grad_fn=<AddBackward0>)
7 tensor(0.7395, grad_fn=<AddBackward0>)
8 tensor(0.7380, grad_fn=<AddBackward0>)
9 tensor(0.7272, grad_fn=<AddBackward0>)
10 tensor(0.7166, grad_fn=<AddBackward0>)
11 tensor(0.7057, grad_fn=<AddBackward0>)
12 tensor(0.6912, grad_fn=<AddBackward0>)
13 tensor(0.6725, grad_fn=<AddBackward0>)
14 tensor(0.6569, grad_fn=<AddBackward0>)
15 tensor(0.6437, grad_fn=<AddBackward0>)
16 tensor(0.6311, grad_fn=<AddBackward0>)
17 tensor(0.6208, grad_fn=<AddBackward0>)
18 tensor(0.6099, grad_fn=<AddBackward0>)
19 tensor(0.6032, grad_fn=<AddBackward0>)
20 tensor(0.5948, grad_fn=<AddBackward0>)
21 tensor(0.5908, grad_fn=<AddBackward0>)
22 tensor(0.5854, grad_fn=<AddBackward0>)
23 tensor(0.5797, grad_fn=<AddBackward0>)
24

2. Compute the mean predictions for 20 MC sampled models

In [11]:
mc_samples = 20
n_test_examples = y_test_np.shape[0]
y_test_mus_z = np.zeros([mc_samples,n_test_examples,1])
for i in range(mc_samples):
    results = model(x_test_t_z)
    y_test_mus_z[i] = results[0].detach().numpy() # Get the predicted means for one sampled parameter vector

3. Compute the Mean Squared Error (MSE) for the variational distribution for the non-linear heteroscedastic regression model for the test data using 20 MC samples and Bayesian model averaging.

In [12]:
y_test_mu_z = np.mean(y_test_mus_z, axis=0) # Compute the mean predictions using Bayesian model averaging
y_test_mu = unscale(y_test_mu_z,y_mu,y_sigma)
print('MSE:', mean_squared_error(y_test_np, y_test_mu))

MSE: 5.681213082439629e-05


4. Compute the epistemic uncertainties for each regression test example.

In [13]:
y_test_mus = unscale(y_test_mus_z,y_mu,y_sigma)
y_test_epistemic = np.mean([(x - y_test_mu)**2 for x in y_test_mus], axis=0) # Compute the epistemic uncertainties

In [14]:
y_test_epistemic

array([[9.71947063e-07],
       [1.30137033e-06],
       [1.41643686e-07],
       ...,
       [5.08810564e-06],
       [1.68738130e-06],
       [7.05133435e-06]], shape=(2984, 1))

5. Select the best test example to add to the training set using epistemic-uncertainity-based active learning

In [15]:
selected_x_test = np.argmax(y_test_epistemic) # Get the index of the best test example to select for active learning

In [16]:
selected_x_test

np.int64(2983)

# Classification for Ship Detection


Load Ship Detection Dataset

In [17]:
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision.io import read_image
from torch.utils.data import random_split
from torchvision.transforms.functional import resize
from sklearn import preprocessing
import numpy as np
from pathlib import Path
import torchmetrics

ROOT_PATH = "shipsnet"
LR = 1e-4
IMG_SIZE = [80]

tensor_size = IMG_SIZE[0]**2 * 3

def max_scaling(image):
    image = image / 255.0
    image = torch.Tensor(image)
    image = resize(image, size=IMG_SIZE)
    return image

def normalize_img(image):
    means = torch.Tensor([[[105.0385]],[[108.1886]],[[ 94.9558]]])
    stds = torch.Tensor([[[48.4294]],[[40.0104]],[[38.6445]]])
    image = torch.Tensor(image)
    image = resize(image, size=IMG_SIZE)
    image = image - means
    image = image / stds
    return image

#https://pytorch.org/tutorials/beginner/basics/data_tutorial.html
class ShipDataset(Dataset):
    def __init__(self, root_path, transform = None):
        self.root_path = Path(root_path)
        self.files = list(self.root_path.rglob("*/*"))
        self.classes = list(set([int(entry.parts[-1]) for entry in self.root_path.rglob("*") if Path(entry).is_dir()]))
        self.transform = transform

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        image = read_image(str(self.files[idx]))
        label = int(self.files[idx].parts[-2])
        if self.transform:
            image = self.transform(image)
        return image, label


full_dataset  = ShipDataset(ROOT_PATH, transform = normalize_img,)
n_classes = len(full_dataset.classes)
n_examples = len(full_dataset)
train_dataset, test_dataset = random_split(full_dataset, [int(0.8*float(n_examples)), int(0.2*float(n_examples))])
n_train_examples = len(train_dataset)
train_dataloader = DataLoader(train_dataset, batch_size=len(train_dataset))
test_dataloader = DataLoader(test_dataset, batch_size=len(test_dataset))

criterion = torch.nn.BCELoss(reduce='mean')
accuracy = torchmetrics.classification.BinaryAccuracy()


C:\Users\shai1\PyCharmProjects\JupyterProject\Bayesian-lab-5\.venv\Lib\site-packages\torch\nn\modules\loss.py:48: UserWarning: size_average and reduce args will be deprecated, please use reduction='mean' instead.
  self.reduction: str = _Reduction.legacy_get_string(size_average, reduce)


In [18]:
n_train_examples = len(train_dataset)

6. Using PyTorch, perform variational inference using Concrete Dropout for a non-linear Bernoulli prediction model for the binary classification dataset

In [3]:
class ConcreteDropout(torch.nn.Module):

    """Concrete Dropout.

    Implementation of the Concrete Dropout module as described in the
    'Concrete Dropout' paper: https://arxiv.org/pdf/1705.07832
    """

    def __init__(self,
                 weight_regulariser: float,
                 dropout_regulariser: float,
                 init_min: float = 0.1,
                 init_max: float = 0.1) -> None:

        """Concrete Dropout.

        Parameters
        ----------
        weight_regulariser : float
            Weight regulariser term.
        dropout_regulariser : float
            Dropout regulariser term.
        init_min : float
            Initial min value.
        init_max : float
            Initial max value.
        """

        super().__init__()

        self.weight_regulariser = weight_regulariser
        self.dropout_regulariser = dropout_regulariser

        init_min = np.log(init_min) - np.log(1.0 - init_min)
        init_max = np.log(init_max) - np.log(1.0 - init_max)

        self.p_logit = torch.nn.parameter.Parameter(torch.empty(1).uniform_(init_min, init_max))
        self.p = torch.sigmoid(self.p_logit)

        self.regularisation = 0.0

    def forward(self, x: torch.Tensor, layer: torch.nn.Module) -> torch.Tensor:

        """Calculates the forward pass.

        The regularisation term for the layer is calculated and assigned to a
        class attribute - this can later be accessed to evaluate the loss.

        Parameters
        ----------
        x : Tensor
            Input to the Concrete Dropout.
        layer : nn.Module
            Layer for which to calculate the Concrete Dropout.

        Returns
        -------
        Tensor
            Output from the dropout layer.
        """

        output = layer(self._concrete_dropout(x))

        sum_of_squares = 0
        for param in layer.parameters():
            sum_of_squares += torch.sum(torch.pow(param, 2))

        weights_reg = self.weight_regulariser * sum_of_squares / (1.0 - self.p)

        dropout_reg = self.p * torch.log(self.p)
        dropout_reg += (1.0 - self.p) * torch.log(1.0 - self.p)
        dropout_reg *= self.dropout_regulariser * x[0].numel()

        self.regularisation = weights_reg + dropout_reg

        return output

    def _concrete_dropout(self, x: torch.Tensor) -> torch.Tensor:

        """Computes the Concrete Dropout.

        Parameters
        ----------
        x : Tensor
            Input tensor to the Concrete Dropout layer.

        Returns
        -------
        Tensor
            Outputs from Concrete Dropout.
        """

        eps = 1e-7
        tmp = 0.1

        self.p = torch.sigmoid(self.p_logit)
        u_noise = torch.rand_like(x)

        drop_prob = (torch.log(self.p + eps) -
                     torch.log(1 - self.p + eps) +
                     torch.log(u_noise + eps) -
                     torch.log(1 - u_noise + eps))

        drop_prob = torch.sigmoid(drop_prob / tmp)

        random_tensor = 1 - drop_prob
        retain_prob = 1 - self.p

        x = torch.mul(x, random_tensor) / retain_prob

        return x

In [19]:
w = 1./(100.*float(n_train_examples))
d = 1./float(n_train_examples)
class nn(torch.nn.Module):
    def __init__(self, inputSize, hiddenSize, outputSize):
        super(nn, self).__init__()
        self.layer1 = torch.nn.Linear(inputSize, hiddenSize)
        self.layer2 = torch.nn.Linear(hiddenSize, hiddenSize)
        self.layer3 = torch.nn.Linear(hiddenSize, outputSize)
        self.cd1 = ConcreteDropout(1e-6, 1e-6) # Call the constructor for ConcreteDropout
        self.cd2 = ConcreteDropout(1e-6, 1e-6) # Call the constructor for ConcreteDropout
        self.cd3 = ConcreteDropout(1e-6, 1e-6) # Call the constructor for ConcreteDropout
        self.relu = torch.nn.ReLU()
    def forward(self, x):
        x = torch.flatten(x, start_dim=1)
        h1 = self.cd1(x, torch.nn.Sequential(self.layer1,self.relu))
        h2 = self.cd2(h1, torch.nn.Sequential(self.layer2,self.relu))
        l = self.cd3(h2, self.layer3)
        return torch.nn.functional.sigmoid(l)

    def reg(self):
        reg = 0.0
        reg += self.cd1.regularisation
        reg += self.cd2.regularisation
        reg += self.cd3.regularisation
        return reg

In [20]:
model = nn(tensor_size, 200, 1)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

for epoch in range(50):
    for data, label in train_dataloader:
        optimizer.zero_grad()
        p = model(data)
        nll_loss = criterion(p.squeeze(),label.float().squeeze())
        loss = nll_loss + model.reg() # Get variational loss for the model
        acc = accuracy(p.squeeze(), label.float().squeeze())
        loss.backward()
        optimizer.step()
        if verbose_option: print(epoch, loss, acc)

0 tensor([0.6624], grad_fn=<AddBackward0>) tensor(0.6909)
1 tensor([0.5719], grad_fn=<AddBackward0>) tensor(0.7756)
2 tensor([0.5203], grad_fn=<AddBackward0>) tensor(0.8012)
3 tensor([0.4952], grad_fn=<AddBackward0>) tensor(0.8134)
4 tensor([0.4690], grad_fn=<AddBackward0>) tensor(0.8125)
5 tensor([0.4422], grad_fn=<AddBackward0>) tensor(0.8169)
6 tensor([0.4209], grad_fn=<AddBackward0>) tensor(0.8272)
7 tensor([0.4037], grad_fn=<AddBackward0>) tensor(0.8391)
8 tensor([0.3908], grad_fn=<AddBackward0>) tensor(0.8459)
9 tensor([0.3744], grad_fn=<AddBackward0>) tensor(0.8575)
10 tensor([0.3560], grad_fn=<AddBackward0>) tensor(0.8703)
11 tensor([0.3400], grad_fn=<AddBackward0>) tensor(0.8775)
12 tensor([0.3240], grad_fn=<AddBackward0>) tensor(0.8778)
13 tensor([0.3110], grad_fn=<AddBackward0>) tensor(0.8863)
14 tensor([0.2991], grad_fn=<AddBackward0>) tensor(0.8925)
15 tensor([0.2850], grad_fn=<AddBackward0>) tensor(0.9003)
16 tensor([0.2723], grad_fn=<AddBackward0>) tensor(0.9094)
17 tens

7.  Compute the predicted probabilities and entropy predictions for 20 MC sampled models

In [27]:
mc_samples = 20
n_test_examples = len(test_dataset)
y_test_probs = np.zeros([mc_samples,n_test_examples,1])
y_test_entropies = np.zeros([mc_samples,n_test_examples,1])
for i in range(mc_samples):
    for data, label in test_dataloader:
        results = model(data).detach().numpy()
        y_test_probs[i] = results # Get the predicted means for one sampled parameter vector
        y_test_entropies[i] += -(results * np.log(results) + (1-results) * np.log(1-results)) # Get the entropy of the predicted probability distributions of the test examples given the sampled parameter vector

8. Compute the Bayesian model averaging predictions for each classification test example.

In [28]:
y_test_probs_avg = np.mean(y_test_probs, axis=0) # Compute Bayesian model averaging predictions

In [29]:
y_test_probs_avg

array([[6.78371753e-01],
       [4.16728184e-06],
       [8.44191900e-02],
       [1.45671004e-03],
       [2.85317268e-06],
       [3.93707165e-03],
       [2.82920055e-02],
       [2.53631293e-03],
       [9.34910670e-01],
       [1.33442618e-01],
       [1.82963214e-03],
       [7.97784211e-03],
       [2.46716252e-01],
       [7.60616452e-01],
       [9.70586610e-01],
       [2.31584168e-02],
       [7.15789163e-02],
       [2.44877561e-04],
       [2.49140459e-04],
       [3.59878520e-03],
       [1.41227548e-05],
       [9.80139163e-01],
       [9.70960784e-01],
       [2.16501398e-02],
       [6.02994881e-06],
       [9.02682513e-01],
       [7.93482251e-04],
       [9.84704372e-01],
       [2.61149947e-05],
       [1.12484399e-02],
       [1.41065993e-03],
       [9.92683652e-01],
       [1.69171468e-03],
       [4.25662860e-01],
       [1.97463784e-01],
       [1.15843222e-01],
       [8.88672422e-04],
       [1.92742104e-02],
       [1.42466761e-01],
       [1.38823214e-05],


9. Compute the aleatoric uncertainty for each classification test example.

In [30]:
y_test_aleatoric  = np.mean(y_test_entropies, axis=0) # Computer aleatoric uncertainty

In [31]:
y_test_aleatoric

array([[5.83323796e-01],
       [5.28324572e-05],
       [2.85765795e-01],
       [9.91777264e-03],
       [3.82726638e-05],
       [2.53924720e-02],
       [1.23767599e-01],
       [1.72077058e-02],
       [2.38183630e-01],
       [3.89560319e-01],
       [1.32479092e-02],
       [4.54843540e-02],
       [5.51518552e-01],
       [5.41091587e-01],
       [1.25379747e-01],
       [1.09353054e-01],
       [2.53624783e-01],
       [2.21929473e-03],
       [2.28839574e-03],
       [2.30119352e-02],
       [1.65972722e-04],
       [9.55837673e-02],
       [1.22595257e-01],
       [1.03680634e-01],
       [7.19488014e-05],
       [3.12956572e-01],
       [6.36829551e-03],
       [7.50534601e-02],
       [2.89477183e-04],
       [5.80970732e-02],
       [1.04384469e-02],
       [4.17104597e-02],
       [1.16554055e-02],
       [6.66952023e-01],
       [4.86729275e-01],
       [3.50919493e-01],
       [6.88291028e-03],
       [9.40777818e-02],
       [4.03994893e-01],
       [1.64475041e-04],


10. Compute the epistemic uncertainty for each classification test example.

In [32]:
y_test_uncertainty = y_test_probs_avg * -1.*np.log(y_test_probs_avg) + (1. - y_test_probs_avg) * -1.*np.log(1. -y_test_probs_avg) # Compute the total uncertainity
y_test_epistemic = y_test_uncertainty - y_test_aleatoric # Compute the epistemic uncertainty

In [33]:
y_test_epistemic

array([[4.47669019e-02],
       [2.96013091e-06],
       [3.66627148e-03],
       [1.05248642e-03],
       [1.00718571e-06],
       [3.37657171e-04],
       [4.98633837e-03],
       [4.85041597e-04],
       [2.56357463e-03],
       [3.31865457e-03],
       [1.13391100e-04],
       [1.00323201e-03],
       [7.18018742e-03],
       [9.27672811e-03],
       [7.31735705e-03],
       [7.35745095e-04],
       [4.07931391e-03],
       [6.16490866e-05],
       [2.79550740e-05],
       [8.31302865e-04],
       [5.86895083e-06],
       [1.91325294e-03],
       [8.79110388e-03],
       [7.12860681e-04],
       [6.55370932e-06],
       [6.19201910e-03],
       [8.96046326e-05],
       [4.06319194e-03],
       [1.22290325e-05],
       [3.56552921e-03],
       [2.30362875e-04],
       [1.55825284e-03],
       [8.31421912e-04],
       [1.51020569e-02],
       [1.01370415e-02],
       [7.64076521e-03],
       [2.48985743e-04],
       [1.12308866e-03],
       [5.42133291e-03],
       [4.67948330e-06],


In [34]:
from sklearn.metrics import classification_report

for data, label in test_dataloader:
    print(classification_report(label.flatten(), y_test_probs_avg.round().flatten()))

              precision    recall  f1-score   support

           0       0.99      0.97      0.98       619
           1       0.91      0.97      0.94       181

    accuracy                           0.97       800
   macro avg       0.95      0.97      0.96       800
weighted avg       0.97      0.97      0.97       800

